# TTT-Dialect: Korean Dialect Speech Recognition

**Before starting:** Runtime -> Change runtime type -> **T4 GPU**

---
**Data upload (one-time setup)**
1. Open `C:\Users\kts64\TTT\data\upload_ready\dialect` on your PC
2. Select all files -> Upload to `TTT-data` folder in [drive.google.com](https://drive.google.com)
3. Once uploaded, run cells below in order

## Step 1 — Mount Google Drive

In [ ]:
# Google Drive 를 Colab 에 연결 (PC 에 있는 파일을 Colab 에서 읽기 위함)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

# Drive 안에 미리 만들어둔 TTT-data 폴더 경로
DATA_DIR = '/content/drive/MyDrive/TTT-data'

# 폴더가 있으면 wav/json 개수 확인, 없으면 Drive 루트 목록 출력
if not os.path.exists(DATA_DIR):
    print('TTT-data folder not found. Drive root contents:')
    for item in os.listdir('/content/drive/MyDrive'):
        print(f'  {item}')
else:
    wav_cnt  = sum(1 for f in os.listdir(DATA_DIR) if f.endswith('.wav'))
    json_cnt = sum(1 for f in os.listdir(DATA_DIR) if f.endswith('.json'))
    print(f'Drive mounted!')
    print(f'  wav: {wav_cnt} / json: {json_cnt}')
    print(f'  path: {DATA_DIR}')

## Step 2 — Clone repo & install packages

In [ ]:
import os

# GitHub 에서 프로젝트 코드 다운로드 (처음 실행 시 clone, 이후엔 pull 로 업데이트)
REPO_DIR = '/content/TTT-Dialect'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kts6450/TTT-Dialect.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

# 작업 디렉토리를 클론한 폴더로 이동
%cd {REPO_DIR}

# 필요한 Python 패키지 설치
# openai-whisper: 베이스 STT 모델
# transformers: 모델 로드/학습 프레임워크
# librosa: 오디오 파일 읽기/처리
# jiwer: WER(단어 오류율) 계산
!pip install -q openai-whisper transformers librosa soundfile jiwer loguru pyyaml accelerate torchaudio
print('Done!')

## Step 3 — Check GPU & Load Model

In [ ]:
import sys, torch, importlib

# 프로젝트 코드를 Python 경로에 추가 (import 가능하게)
sys.path.insert(0, '/content/TTT-Dialect')

# GPU 사용 가능 여부 확인 (T4 GPU 가 있어야 빠르게 학습 가능)
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  {torch.cuda.get_device_name(0)}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# 모델 코드 최신으로 리로드 (git pull 후 반영 위해)
import models.base_whisper as bw
importlib.reload(bw)
from models.base_whisper import KoreanWhisperModel

# HuggingFace 에 공개된 한국어 Whisper 모델 로드
# (OpenAI Whisper 를 한국어로 파인튜닝한 버전)
MODEL_NAME = 'SungBeom/whisper-small-ko'
print(f'Loading model: {MODEL_NAME}')
model = KoreanWhisperModel(MODEL_NAME)
model.model = model.model.to(DEVICE)  # 모델을 GPU 로 이동
params = model.count_parameters()
print(f'Done! Parameters: {params["total"]:,}')

## Step 4 — Build manifest from AI Hub dialect data

In [ ]:
import json, os, librosa
from tqdm.notebook import tqdm

SR = 16000      # 샘플링 레이트 16kHz (Whisper 표준)
manifest = []   # 전체 샘플 목록 (오디오 경로 + 전사 텍스트)

# AI Hub 지역 코드 -> 방언 이름 매핑
PROVINCE_MAP = {
    'gs': 'Gyeongsang', 'gw': 'Gangwon', 'jl': 'Jeolla',
    'cc': 'Chungcheong', 'jj': 'Jeju', 'se': 'Seoul'
}

def parse_label(json_path):
    # AI Hub json 라벨 파일에서 전사 텍스트, 방언, 나이 추출
    with open(json_path, encoding='utf-8') as jf:
        d = json.load(jf)
    trans = d.get('transcription', {})

    # 표준어 전사 텍스트 가져오기 (standard 필드)
    text = trans.get('standard', '').strip()

    # standard 가 없으면 sentences 리스트에서 이어붙이기
    if not text:
        sents = trans.get('sentences', [])
        text = ' '.join(
            s.get('standard', '') or s.get('dialect', '')
            for s in sents
        ).strip()

    # 화자 정보에서 방언 지역 코드와 출생연도 추출
    speakers = d.get('speaker', [{}])
    province_code = speakers[0].get('residenceProvince', 'gs') if speakers else 'gs'
    dialect = PROVINCE_MAP.get(province_code, province_code)
    try:
        age = 2024 - int(speakers[0].get('birthYear', 1960))
    except:
        age = 65
    return text, dialect, age

# Drive 폴더에서 wav / json 파일 목록 수집
files = os.listdir(DATA_DIR)
wav_files  = [f for f in files if f.endswith('.wav')]
json_stems = {os.path.splitext(f)[0] for f in files if f.endswith('.json')}

print(f'Processing {len(wav_files)} wav files...')
added, skip = 0, 0

for wav_fname in tqdm(wav_files):
    try:
        stem = os.path.splitext(wav_fname)[0]  # 확장자 제거한 파일명

        # 같은 이름의 json 이 없으면 건너뜀
        if stem not in json_stems:
            skip += 1
            continue

        json_path = os.path.join(DATA_DIR, stem + '.json')
        wav_path  = os.path.join(DATA_DIR, wav_fname)

        text, dialect, age = parse_label(json_path)
        if not text:  # 전사 텍스트가 없으면 건너뜀
            skip += 1
            continue

        # 오디오 로드 후 길이 필터링 (너무 짧거나 긴 파일 제외)
        audio = librosa.load(wav_path, sr=SR)[0]
        duration = len(audio) / SR
        if duration < 0.5 or duration > 60:
            skip += 1
            continue

        manifest.append({
            'audio_path': wav_path,
            'transcript': text,
            'dialect': dialect,
            'speaker_age': age,
            'speaker_id': stem[:12],
            'duration_sec': round(duration, 2),
            'type': 'dialect'
        })
        added += 1
    except:
        skip += 1

# manifest 를 jsonl 파일로 저장 (한 줄 = 한 샘플)
MANIFEST_PATH = '/content/manifest.jsonl'
with open(MANIFEST_PATH, 'w', encoding='utf-8') as f:
    for m in manifest:
        f.write(json.dumps(m, ensure_ascii=False) + '\n')

print(f'added: {added} / skipped: {skip}')
if manifest:
    print(f'sample transcript: "{manifest[0]["transcript"][:60]}"')
    print(f'dialect: {manifest[0]["dialect"]} | age: {manifest[0]["speaker_age"]}')
else:
    print('0 samples — run diagnostic cell below')

## Step 4-Diagnostic (run only if manifest = 0)

In [ ]:
import json, os
for f in os.listdir(DATA_DIR):
    if f.endswith('.json'):
        with open(os.path.join(DATA_DIR, f), encoding='utf-8') as jf:
            d = json.load(jf)
        print('top keys:', list(d.keys()))
        trans = d.get('transcription', {})
        print('transcription keys:', list(trans.keys()))
        print('standard:', str(trans.get('standard', ''))[:100])
        sents = trans.get('sentences', [])
        if sents:
            print('sentences[0] keys:', list(sents[0].keys()))
            print('sentences[0].standard:', sents[0].get('standard', ''))
        break

## Step 5 — Initialize TTT Adapter

In [ ]:
import importlib
import models.ttt_adapter as ta
importlib.reload(ta)
from models.ttt_adapter import TTTAdapter

# TTT 어댑터 초기화
# top_k_layers=2  : Whisper 인코더의 상위 2개 레이어만 업데이트 (나머지는 고정)
# lr=1e-4         : 학습률 (너무 크면 모델이 망가지고, 너무 작으면 적응 안 됨)
# adaptation_steps: 캘리브레이션 음성 1개당 gradient update 횟수
adapter = TTTAdapter(
    base_model=model,
    profile_dir='/content/user_profiles',  # 사용자별 적응 결과 저장 폴더
    top_k_layers=2,
    lr=1e-4,
    adaptation_steps=30,
)
print('TTT adapter ready!')

## Step 6 — Measure WER before TTT

In [ ]:
import json, librosa, torch
from jiwer import wer
from tqdm.notebook import tqdm

SR = 16000
N_TEST = 50  # WER 측정에 사용할 테스트 샘플 수

# manifest 파일에서 전체 샘플 로드
samples = []
with open(MANIFEST_PATH) as f:
    for line in f:
        samples.append(json.loads(line))

print(f'Total samples: {len(samples)}')
if len(samples) == 0:
    raise SystemExit('manifest is empty — re-run Step 4')

# 앞 50개를 테스트용으로 고정
test_samples = samples[:N_TEST]

def infer(sample, mdl):
    # 오디오 파일을 읽어서 Whisper 입력 형식으로 변환 후 텍스트 출력
    audio, _ = librosa.load(sample['audio_path'], sr=SR)
    feat = mdl.processor.feature_extractor(
        audio, sampling_rate=SR, return_tensors='pt'
    ).input_features.to(DEVICE)
    with torch.no_grad():  # 추론 시엔 gradient 계산 불필요
        result = mdl.transcribe(feat)
    return result[0] if result else ''

# TTT 적용 전 WER 측정 (베이스라인)
print(f'Measuring WER before TTT ({N_TEST} samples)...')
refs, hyps = [], []
for s in tqdm(test_samples):
    refs.append(s['transcript'])   # 정답 텍스트
    hyps.append(infer(s, model))   # 모델이 인식한 텍스트

wer_before = wer(refs, hyps)
print(f'WER before TTT: {wer_before:.1%}')
print(f'  ref:  "{refs[0][:50]}"')   # 정답 예시
print(f'  hyp:  "{hyps[0][:50]}"')   # 인식 결과 예시

## Step 7 — TTT Calibration (core)

In [ ]:
# 캘리브레이션에 쓸 샘플 수 (테스트 샘플 이후부터 사용)
N_CALIB = 20
calib_samples = samples[N_TEST:N_TEST + N_CALIB]

# 캘리브레이션용 오디오 특징 추출
calib_features, calib_texts = [], []
for s in calib_samples:
    audio, _ = librosa.load(s['audio_path'], sr=SR)
    feat = model.processor.feature_extractor(
        audio, sampling_rate=SR, return_tensors='pt'
    ).input_features[0]  # [0]: 배치 차원 제거
    calib_features.append(feat)
    calib_texts.append(s['transcript'])

print(f'TTT calibration with {N_CALIB} samples (~1-2 min)...')

# 핵심: TTT 캘리브레이션 실행
# 이 과정에서 모델의 상위 2개 레이어가 이 화자의 음성에 맞게 업데이트됨
profile = adapter.calibrate(
    user_id='aihub_dialect_user',      # 사용자 ID (프로파일 저장에 사용)
    audio_features=calib_features,     # 캘리브레이션 오디오 특징
    transcripts=calib_texts,           # 정답 텍스트 (지도 학습)
    dialect=calib_samples[0].get('dialect', 'Gyeongsang'),
    age=calib_samples[0].get('speaker_age', 65),
)

print(f'TTT done!')
print(f'  WER before: {profile.wer_before:.1%}')   # TTT 전 WER
print(f'  WER after:  {profile.wer_after:.1%}')    # TTT 후 WER
print(f'  Improvement: {profile.wer_before - profile.wer_after:.1%}p')  # 개선량

## Step 8 — Visualize results & save to Drive

In [ ]:
import matplotlib.pyplot as plt, json, os

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('TTT Performance Comparison (AI Hub Dialect Data)', fontsize=14, fontweight='bold')

ax = axes[0]
values = [profile.wer_before * 100, profile.wer_after * 100]
bars = ax.bar(['Before TTT', 'After TTT'], values, color=['#FF6B6B', '#4ECDC4'], alpha=0.85, width=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontweight='bold', fontsize=13)
ax.set_ylabel('WER (%) — lower is better')
ax.set_title('WER Comparison')
ax.set_ylim(0, max(values) * 1.3)
ax.spines[['top', 'right']].set_visible(False)

ax2 = axes[1]
if profile.adaptation_history:
    steps = list(range(len(profile.adaptation_history)))
    wers = [h['wer'] * 100 for h in profile.adaptation_history]
    ax2.plot(steps, wers, 'o-', color='#667eea', linewidth=2.5, markersize=8)
    ax2.fill_between(steps, wers, alpha=0.15, color='#667eea')
    ax2.set_xlabel('Adaptation step')
    ax2.set_ylabel('WER (%)')
    ax2.set_title('Adaptation History')
    ax2.spines[['top', 'right']].set_visible(False)
plt.tight_layout()

RESULT_DIR = '/content/drive/MyDrive/TTT-results'
os.makedirs(RESULT_DIR, exist_ok=True)
plt.savefig(f'{RESULT_DIR}/wer_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

summary = {
    'model': MODEL_NAME,
    'data': 'AI Hub Dialect (Gyeongsang + Gangwon)',
    'n_test': N_TEST, 'n_calib': N_CALIB,
    'wer_before': profile.wer_before,
    'wer_after': profile.wer_after,
    'improvement_pp': profile.wer_before - profile.wer_after,
    'improvement_pct': (profile.wer_before - profile.wer_after) / max(profile.wer_before, 1e-9) * 100
}
with open(f'{RESULT_DIR}/summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f'WER before: {summary["wer_before"]:.1%}')
print(f'WER after:  {summary["wer_after"]:.1%}')
print(f'Improvement: {summary["improvement_pp"]:.1%}p ({summary["improvement_pct"]:.1f}%)')
print(f'Saved to: {RESULT_DIR}')

## Step 9 — Ablation: WER vs Calibration Sample Count (paper figure)

In [ ]:
import json, librosa, torch, copy, os
import matplotlib.pyplot as plt
from jiwer import wer as calc_wer
from tqdm.notebook import tqdm
import importlib
import models.ttt_adapter as ta
importlib.reload(ta)
from models.ttt_adapter import TTTAdapter

SR = 16000
N_TEST = 50

# 실험할 캘리브레이션 샘플 수 목록 (0 = TTT 없음, 즉 베이스라인)
CALIB_COUNTS = [0, 5, 10, 20, 50]

results = []
test_samples = samples[:N_TEST]    # 테스트용 50개 (항상 고정)
pool_samples = samples[N_TEST:]    # 캘리브레이션 풀 (테스트와 겹치지 않게)

def infer_batch(sample_list, mdl):
    # 샘플 리스트 전체를 추론하고 WER 반환
    refs, hyps = [], []
    for s in sample_list:
        audio, _ = librosa.load(s['audio_path'], sr=SR)
        feat = mdl.processor.feature_extractor(
            audio, sampling_rate=SR, return_tensors='pt'
        ).input_features.to(DEVICE)
        with torch.no_grad():
            out = mdl.transcribe(feat)
        refs.append(s['transcript'])
        hyps.append(out[0] if out else '')
    return calc_wer(refs, hyps)

print('Measuring WER for each calibration count...')
print('(~5-10 min total)\n')

for n_calib in CALIB_COUNTS:
    print(f'[n_calib={n_calib}] ', end='', flush=True)

    # 매 실험마다 모델을 초기 상태로 복사 (독립 실험 보장)
    fresh_model = copy.deepcopy(model)
    fresh_model.model = fresh_model.model.to(DEVICE)

    if n_calib == 0:
        # 베이스라인: TTT 없이 바로 WER 측정
        wer_val = infer_batch(test_samples, fresh_model)
        print(f'WER={wer_val:.1%} (baseline, no TTT)')
    else:
        # n_calib 개 샘플로 TTT 캘리브레이션 후 WER 측정
        calib_s = pool_samples[:n_calib]
        calib_features, calib_texts = [], []
        for s in calib_s:
            audio, _ = librosa.load(s['audio_path'], sr=SR)
            feat = fresh_model.processor.feature_extractor(
                audio, sampling_rate=SR, return_tensors='pt'
            ).input_features[0]
            calib_features.append(feat)
            calib_texts.append(s['transcript'])

        fresh_adapter = TTTAdapter(
            base_model=fresh_model,
            profile_dir=f'/content/profiles_{n_calib}',
            top_k_layers=2, lr=1e-4, adaptation_steps=30,
        )
        prof = fresh_adapter.calibrate(
            user_id=f'exp_{n_calib}',
            audio_features=calib_features,
            transcripts=calib_texts,
            dialect=calib_s[0].get('dialect', 'Gyeongsang'),
            age=65,
        )
        wer_val = prof.wer_after  # TTT 적용 후 WER
        print(f'WER={wer_val:.1%}')

    results.append({'n_calib': n_calib, 'wer': wer_val})

# 결과 요약 출력
print('\n=== Results ===')
baseline_wer = results[0]['wer']
for r in results:
    tag = '(baseline)' if r['n_calib'] == 0 else f"(improvement: {(baseline_wer - r['wer'])*100:.1f}pp)"
    print(f"  n={r['n_calib']:2d}: WER {r['wer']:.1%}  {tag}")

# 그래프 그리기
fig, ax = plt.subplots(figsize=(9, 5))
ns   = [r['n_calib'] for r in results]
wers = [r['wer'] * 100 for r in results]

ax.plot(ns, wers, 'o-', color='#667eea', linewidth=2.5, markersize=10, zorder=3)
ax.fill_between(ns, wers, alpha=0.12, color='#667eea')

# 각 점 위에 WER 수치 표시
for n, w in zip(ns, wers):
    ax.annotate(f'{w:.1f}%', (n, w), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=11, fontweight='bold')

# 베이스라인 수평선
ax.axhline(y=baseline_wer * 100, color='#FF6B6B', linestyle='--',
           linewidth=1.5, label=f'Baseline (no TTT): {baseline_wer:.1%}')
ax.set_xlabel('Number of calibration samples', fontsize=12)
ax.set_ylabel('WER (%) — lower is better', fontsize=12)
ax.set_title('WER vs Calibration Sample Count', fontsize=14, fontweight='bold')
ax.set_xticks(ns)
ax.set_xticklabels([str(n) if n > 0 else '0\n(baseline)' for n in ns])
ax.legend(fontsize=11)
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()

# Drive 에 그래프와 json 결과 저장
RESULT_DIR = '/content/drive/MyDrive/TTT-results'
os.makedirs(RESULT_DIR, exist_ok=True)
plt.savefig(f'{RESULT_DIR}/calib_count_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

with open(f'{RESULT_DIR}/calib_count_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'Saved: {RESULT_DIR}/calib_count_analysis.png')